# Day 1: Data Preprocessing and Feature Engineering

This notebook is intentionally focused on **data preparation and feature engineering** (no model benchmarking).
The main flow uses `adult_income_issues.csv`, then we transfer the same reasoning to Ames and Retail mini-labs.

---

### Table of Contents

- [0. Scope and Success Criteria](#scope)
- [1. Data Intake and Baseline Audit](#audit)
- [2. Splitting Strategy and Leakage Awareness](#splitting)
- [3. Categorical Encoding](#encoding)
- [4. Missing Values](#missing)
- [5. Invalid Values and Outliers](#outliers)
- [6. Scaling and Numeric Transformations](#scaling)
- [7. Feature Engineering](#engineering)
- [8. Feature Selection](#selection)
- [9. Mini-Lab A (Ames)](#ames-lab)
- [10. Mini-Lab B (Retail)](#retail-lab)
- [11. Preprocessing Checklist](#checklist)
- [Acceptance Checks](#acceptance)

<a id="scope"></a>
## Section 0 - Scope and Success Criteria

**Scope for Day 1**
- Data audit and preprocessing by category.
- Leakage-safe transformation design.
- Feature engineering and qualitative feature selection.
- No model-comparison section (reserved for Day 2).

**Success criteria**
- You can identify preprocessing needs by category (not by manual column-by-column trial and error).
- You can fit transformations on train only and apply them to val/test safely.
- You can explain tradeoffs across encoding, imputation, outlier handling, scaling, and feature design.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PowerTransformer,
    KBinsDiscretizer,
    PolynomialFeatures,
)
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.feature_extraction import FeatureHasher

# IterativeImputer is experimental in sklearn.
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

try:
    import category_encoders as ce
    HAS_CATEGORY_ENCODERS = True
except Exception:
    HAS_CATEGORY_ENCODERS = False


def show(df, n=5):
    display(df.head(n))


def to_numeric_loose(series: pd.Series) -> pd.Series:
    # Handles values like "40h" while preserving NaN for unparseable values.
    cleaned = series.astype("string").str.strip().str.replace("h", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")


print("Environment ready. category_encoders installed:", HAS_CATEGORY_ENCODERS)


<a id="audit"></a>
## Section 1 - Data Intake and Baseline Audit (Adult main)

In [ ]:
adult = pd.read_csv("day1/generated/adult_income_issues.csv")
ames = pd.read_csv("day1/generated/ames_housing_issues.csv")
retail = pd.read_csv("day1/generated/retail_panel_issues.csv")

TARGET_COL = "class"
TARGET_BIN_COL = "target"
SPLIT_COL = "split"
ID_COLS = ["person_id"]

LEAKAGE_COLS = [
    "post_adjudication_risk_code",  # explicitly post-outcome style feature
]

PROCESS_COLS = [
    "db_source_table",
    "db_etl_batch_id",
    "db_row_surrogate_key",
    "db_loaded_at_utc",
    "dataset_schema_version",
    "extract_country_code",
    "record_written_at",
    "dgp_regime",
]

adult[TARGET_BIN_COL] = adult[TARGET_COL].astype(str).str.contains(">50", case=False, regex=False).astype(int)

print("Adult shape:", adult.shape)
print("Ames shape:", ames.shape)
print("Retail shape:", retail.shape)
print("Adult target positive rate:", round(adult[TARGET_BIN_COL].mean(), 4))


In [ ]:
def build_audit_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        sample_vals = s.dropna().astype(str).head(3).tolist()
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "pct_missing": float(s.isna().mean()),
            "n_unique": int(s.nunique(dropna=True)),
            "example_values": sample_vals,
        })
    out = pd.DataFrame(rows)
    out["suspicious"] = (
        out["column"].str.contains("risk|post|etl|db_|loaded|schema|surrogate|target", case=False, regex=True)
        | out["column"].str.endswith("_flag")
    )
    return out.sort_values(["suspicious", "pct_missing", "n_unique"], ascending=[False, False, False])


audit_adult = build_audit_table(adult)
show(audit_adult, n=20)


> **Do It Yourself**
>
> Classify columns into: `numeric`, `categorical`, `text`, `date`, `process/metadata`, `target/id`.
>
> Hint: use both dtype checks and domain clues from column names.

In [ ]:
column_roles = []
for col in adult.columns:
    s = adult[col]
    lower = col.lower()

    if col in [TARGET_COL, TARGET_BIN_COL] or col in ID_COLS:
        role = "target_or_id"
    elif col in PROCESS_COLS or lower.startswith("db_") or lower.endswith("_at_utc"):
        role = "process_metadata"
    elif "date" in lower or "time" in lower or "written_at" in lower:
        role = "date"
    elif s.dtype.kind in "biufc":
        role = "numeric"
    else:
        as_num = to_numeric_loose(s)
        numeric_ratio = as_num.notna().mean()
        avg_len = s.astype("string").str.len().fillna(0).mean()
        nunique = s.nunique(dropna=True)

        if numeric_ratio >= 0.9:
            role = "numeric"
        elif avg_len > 30:
            role = "text"
        elif nunique <= max(100, int(0.2 * len(s))):
            role = "categorical"
        else:
            role = "text"

    use_for_modeling = role in {"numeric", "categorical", "date"} and col not in LEAKAGE_COLS
    column_roles.append({"column": col, "role": role, "use_for_modeling": use_for_modeling})

column_roles = pd.DataFrame(column_roles).sort_values(["use_for_modeling", "role", "column"], ascending=[False, True, True])
show(column_roles, n=40)

model_candidate_cols = column_roles.loc[column_roles["use_for_modeling"], "column"].tolist()
print("Model candidate columns:", len(model_candidate_cols))


<a id="splitting"></a>
## Section 2 - Splitting Strategy and Leakage Awareness

We use the dataset `split` column when available. This preserves the injected "new categories at test time" behavior.
Then we derive a validation split from train.

In [ ]:
if SPLIT_COL in adult.columns:
    train_pool = adult.loc[adult[SPLIT_COL] == "train"].copy()
    test_df = adult.loc[adult[SPLIT_COL] == "test"].copy()
else:
    train_pool, test_df = train_test_split(
        adult,
        test_size=0.2,
        random_state=SEED,
        stratify=adult[TARGET_BIN_COL],
    )

# In this synthetic issues dataset, duplicate/conflicting IDs can leak across splits.
# We enforce entity-level separation by removing overlapping IDs from train pool.
overlap_ids = set(train_pool[ID_COLS[0]]) & set(test_df[ID_COLS[0]])
if overlap_ids:
    train_pool = train_pool.loc[~train_pool[ID_COLS[0]].isin(overlap_ids)].copy()
    print(f"Removed {len(overlap_ids)} overlapping IDs from train pool to enforce split integrity.")

# Entity-aware train/val split: split IDs, not rows.
id_series = train_pool[ID_COLS[0]].dropna()
id_target = train_pool.groupby(ID_COLS[0])[TARGET_BIN_COL].mean().round().astype(int)
strat = id_target.values if id_target.nunique() > 1 else None

id_train, id_val = train_test_split(
    id_target.index.to_numpy(),
    test_size=0.2,
    random_state=SEED,
    stratify=strat,
)

train_df = train_pool.loc[train_pool[ID_COLS[0]].isin(id_train)].copy()
val_df = train_pool.loc[train_pool[ID_COLS[0]].isin(id_val)].copy()

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5} rows={len(frame):5d} target_rate={frame[TARGET_BIN_COL].mean():.4f}")

train_ids = set(train_df[ID_COLS[0]])
val_ids = set(val_df[ID_COLS[0]])
test_ids = set(test_df[ID_COLS[0]])

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

print("Split integrity check passed (no ID overlap across train/val/test).")


In [ ]:
def unseen_categories(train_series: pd.Series, other_series: pd.Series) -> list:
    a = set(train_series.dropna().astype(str).unique())
    b = set(other_series.dropna().astype(str).unique())
    return sorted(b - a)

cat_cols_for_check = [c for c in model_candidate_cols if c in adult.columns and adult[c].dtype == "object"]

if "occupation" in adult.columns:
    unseen_occ_test = unseen_categories(train_df["occupation"], test_df["occupation"])
    print("Unseen occupation categories in test vs train:", unseen_occ_test)
    assert len(unseen_occ_test) > 0, "Expected at least one unseen category in test."

split_summary = (
    adult.groupby(SPLIT_COL)[TARGET_BIN_COL]
    .agg(["count", "mean"])
    .rename(columns={"mean": "target_rate"})
)
show(split_summary.reset_index(), n=10)


> **Do It Yourself**
>
> 1. Check if split proportions and target prevalence are reasonable.
> 2. Find at least one categorical feature where test has unseen labels compared to train.
>
> Hint: compare `set(test[col]) - set(train[col])`.

In [ ]:
split_prop = adult[SPLIT_COL].value_counts(normalize=True).rename("share").reset_index().rename(columns={"index": SPLIT_COL})
show(split_prop, n=10)

for col in ["occupation", "native_country", "workclass"]:
    if col in adult.columns:
        unseen = unseen_categories(train_df[col], test_df[col])
        print(f"{col}: unseen in test={len(unseen)} -> {unseen[:8]}")

print("Bad split examples to avoid: group leakage (same entity in train/test), and time leakage (future rows in train).")


<a id="encoding"></a>
## Section 3 - Categorical Encoding (Popular Methods)

We compare label, ordinal, one-hot, frequency, and target encoding.
All mappings/encoders are learned on **train only**.

In [ ]:
excluded_cols = set([TARGET_COL, TARGET_BIN_COL, SPLIT_COL] + ID_COLS + LEAKAGE_COLS + PROCESS_COLS)
feature_cols = [c for c in adult.columns if c not in excluded_cols]

# Numeric-like columns can still be object due to mixed types (e.g., "40h").
numeric_like_cols = []
categorical_cols = []
for col in feature_cols:
    s = train_df[col]
    if s.dtype.kind in "biufc":
        numeric_like_cols.append(col)
        continue
    as_num = to_numeric_loose(s)
    if as_num.notna().mean() >= 0.9:
        numeric_like_cols.append(col)
    else:
        categorical_cols.append(col)

# Keep short text comments out of default categorical encoding demos.
categorical_cols = [c for c in categorical_cols if c not in ["case_review_note"]]

X_train_raw = train_df[feature_cols].copy()
X_val_raw = val_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

y_train = train_df[TARGET_BIN_COL].copy()
y_val = val_df[TARGET_BIN_COL].copy()
y_test = test_df[TARGET_BIN_COL].copy()

print("Feature columns:", len(feature_cols))
print("Numeric-like columns:", len(numeric_like_cols))
print("Categorical columns:", len(categorical_cols))


In [ ]:
# Label Encoding (manual map) - demo on occupation
label_maps = {}
X_train_label = pd.DataFrame(index=X_train_raw.index)
X_val_label = pd.DataFrame(index=X_val_raw.index)
X_test_label = pd.DataFrame(index=X_test_raw.index)

if "occupation" in X_train_raw.columns:
    col = "occupation"
    labels = sorted(X_train_raw[col].dropna().astype(str).unique())
    label_maps[col] = {v: i for i, v in enumerate(labels)}

    X_train_label[f"{col}_label"] = X_train_raw[col].astype(str).map(label_maps[col]).fillna(-1).astype(int)
    X_val_label[f"{col}_label"] = X_val_raw[col].astype(str).map(label_maps[col]).fillna(-1).astype(int)
    X_test_label[f"{col}_label"] = X_test_raw[col].astype(str).map(label_maps[col]).fillna(-1).astype(int)

    print("Label map sample:", list(label_maps[col].items())[:8])
    print("Unknown mapping count in test:", int((X_test_label[f"{col}_label"] == -1).sum()))

show(X_test_label, n=5)


In [ ]:
# Ordinal Encoding - only for ordered categories.
# We create an ordered feature from hours_per_week for demonstration.

ordinal_maps = {}

def build_hours_band(df: pd.DataFrame) -> pd.Series:
    hours = to_numeric_loose(df["hours_per_week"])
    return pd.cut(
        hours,
        bins=[-np.inf, 30, 45, 60, np.inf],
        labels=["very_low", "low", "medium", "high"],
        ordered=True,
    )

for frame in [X_train_raw, X_val_raw, X_test_raw]:
    frame["hours_band"] = build_hours_band(frame)

ord_col = "hours_band"
ord_categories = [["very_low", "low", "medium", "high"]]
ord_enc = OrdinalEncoder(categories=ord_categories, handle_unknown="use_encoded_value", unknown_value=-1)

X_train_ord = ord_enc.fit_transform(X_train_raw[[ord_col]])
X_val_ord = ord_enc.transform(X_val_raw[[ord_col]])
X_test_ord = ord_enc.transform(X_test_raw[[ord_col]])

ordinal_maps[ord_col] = {c: i for i, c in enumerate(ord_categories[0])}

print("Ordinal map:", ordinal_maps[ord_col])
print("Train ordinal sample:", X_train_ord[:10].ravel().tolist())


In [ ]:
# One-Hot Encoding for nominal categories
nominal_cols = [c for c in ["occupation", "workclass", "marital_status", "relationship", "native_country"] if c in X_train_raw.columns]

if nominal_cols:
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    train_nom = X_train_raw[nominal_cols].fillna("MISSING").astype(str)
    val_nom = X_val_raw[nominal_cols].fillna("MISSING").astype(str)
    test_nom = X_test_raw[nominal_cols].fillna("MISSING").astype(str)

    X_train_ohe = ohe.fit_transform(train_nom)
    X_val_ohe = ohe.transform(val_nom)
    X_test_ohe = ohe.transform(test_nom)

    ohe_cols = ohe.get_feature_names_out(nominal_cols)
    X_train_ohe_df = pd.DataFrame(X_train_ohe, index=X_train_raw.index, columns=ohe_cols)
    X_val_ohe_df = pd.DataFrame(X_val_ohe, index=X_val_raw.index, columns=ohe_cols)
    X_test_ohe_df = pd.DataFrame(X_test_ohe, index=X_test_raw.index, columns=ohe_cols)

    print("OHE shapes:", X_train_ohe_df.shape, X_val_ohe_df.shape, X_test_ohe_df.shape)
    show(X_train_ohe_df, n=3)
else:
    X_train_ohe_df = pd.DataFrame(index=X_train_raw.index)
    X_val_ohe_df = pd.DataFrame(index=X_val_raw.index)
    X_test_ohe_df = pd.DataFrame(index=X_test_raw.index)


In [ ]:
# Frequency Encoding
freq_maps = {}
X_train_freq = pd.DataFrame(index=X_train_raw.index)
X_val_freq = pd.DataFrame(index=X_val_raw.index)
X_test_freq = pd.DataFrame(index=X_test_raw.index)

freq_cols = [c for c in ["occupation", "native_country"] if c in X_train_raw.columns]
for col in freq_cols:
    freq = X_train_raw[col].astype(str).value_counts(normalize=True)
    freq_maps[col] = freq
    X_train_freq[f"{col}_freq"] = X_train_raw[col].astype(str).map(freq).fillna(0.0)
    X_val_freq[f"{col}_freq"] = X_val_raw[col].astype(str).map(freq).fillna(0.0)
    X_test_freq[f"{col}_freq"] = X_test_raw[col].astype(str).map(freq).fillna(0.0)

show(X_train_freq, n=5)


In [ ]:
# Target Encoding: wrong (leaky) vs safer train-only + OOF idea

def oof_target_encode(train_cat: pd.Series, y: pd.Series, n_splits: int = 5, seed: int = 42):
    train_cat = train_cat.astype(str)
    y = y.astype(float)
    global_mean = y.mean()

    oof = pd.Series(index=train_cat.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr_idx, ho_idx in kf.split(train_cat):
        tr_c = train_cat.iloc[tr_idx]
        tr_y = y.iloc[tr_idx]
        fold_map = tr_y.groupby(tr_c).mean()
        oof.iloc[ho_idx] = train_cat.iloc[ho_idx].map(fold_map).fillna(global_mean)

    full_map = y.groupby(train_cat).mean()
    return oof, full_map, global_mean


target_maps = {}
if "occupation" in X_train_raw.columns:
    te_col = "occupation"

    # Wrong/leaky: uses full dataset (including val/test information)
    leaky_map = adult.groupby(adult[te_col].astype(str))[TARGET_BIN_COL].mean()
    train_te_leaky = X_train_raw[te_col].astype(str).map(leaky_map).fillna(adult[TARGET_BIN_COL].mean())

    # Safer: train-only + OOF
    train_te_oof, safe_map, safe_global = oof_target_encode(X_train_raw[te_col], y_train, n_splits=5, seed=SEED)
    val_te_safe = X_val_raw[te_col].astype(str).map(safe_map).fillna(safe_global)
    test_te_safe = X_test_raw[te_col].astype(str).map(safe_map).fillna(safe_global)

    target_maps[te_col] = safe_map

    compare = pd.DataFrame({
        "y_train": y_train,
        "te_leaky": train_te_leaky,
        "te_safe_oof": train_te_oof,
    })
    print("Train correlation with target (leaky TE):", round(compare[["y_train", "te_leaky"]].corr().iloc[0, 1], 4))
    print("Train correlation with target (safe OOF TE):", round(compare[["y_train", "te_safe_oof"]].corr().iloc[0, 1], 4))

    show(compare, n=5)


In [ ]:
# Optional library-based encoding path (category_encoders)
if HAS_CATEGORY_ENCODERS and "occupation" in X_train_raw.columns:
    ce_te = ce.TargetEncoder(cols=["occupation"])
    train_ce = ce_te.fit_transform(X_train_raw[["occupation"]].fillna("MISSING"), y_train)
    val_ce = ce_te.transform(X_val_raw[["occupation"]].fillna("MISSING"))
    print("category_encoders TargetEncoder output sample:")
    show(train_ce, n=5)
else:
    print("category_encoders not installed; manual path above is the required baseline.")


> **Do It Yourself**
>
> For each categorical feature, pick **one encoding method** and justify why:
> - label / ordinal / one-hot / frequency / target encoding.
>
> Hint: decide first whether the category is ordered and whether unseen labels are likely at inference time.

In [ ]:
# Reference solution: a practical mixed strategy
numeric_df_train = pd.DataFrame(index=X_train_raw.index)
numeric_df_val = pd.DataFrame(index=X_val_raw.index)
numeric_df_test = pd.DataFrame(index=X_test_raw.index)

for col in numeric_like_cols:
    numeric_df_train[col] = to_numeric_loose(X_train_raw[col])
    numeric_df_val[col] = to_numeric_loose(X_val_raw[col])
    numeric_df_test[col] = to_numeric_loose(X_test_raw[col])

mixed_train = pd.concat([
    numeric_df_train,
    X_train_ohe_df,
    X_train_freq,
], axis=1)
mixed_val = pd.concat([
    numeric_df_val,
    X_val_ohe_df,
    X_val_freq,
], axis=1)
mixed_test = pd.concat([
    numeric_df_test,
    X_test_ohe_df,
    X_test_freq,
], axis=1)

# Add safe target encoding feature for one high-cardinality column.
if "occupation" in X_train_raw.columns:
    mixed_train["occupation_te_safe"] = train_te_oof
    mixed_val["occupation_te_safe"] = val_te_safe
    mixed_test["occupation_te_safe"] = test_te_safe

print("Mixed encoded shapes:", mixed_train.shape, mixed_val.shape, mixed_test.shape)
assert mixed_train.shape[1] == mixed_val.shape[1] == mixed_test.shape[1]


<a id="missing"></a>
## Section 4 - Missing Values (Full Ladder)

Methods covered from simple to advanced:
- drop rows/columns
- constant fill
- median/mode
- missing indicators
- group-wise imputation
- KNN imputation
- Iterative (MICE-like) imputation

In [ ]:
missing_report = (
    train_df.isna().mean().rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
    .sort_values("missing_pct", ascending=False)
)
show(missing_report, n=15)

# Example cues for missingness mechanisms
if "sale_price" in ames.columns and "reported_max_loan" in ames.columns:
    tmp = ames.assign(sale_price_missing=ames["sale_price"].isna())
    print("Ames loan stats by sale_price missingness:")
    display(tmp.groupby("sale_price_missing")["reported_max_loan"].describe())

print("Interpretation reminder: MCAR/MAR/MNAR is a data-generating assumption, not a test you prove with certainty.")


In [ ]:
missing_demo_cols = [c for c in ["age", "hours_per_week", "capital_gain", "occupation"] if c in train_df.columns]
missing_demo = train_df[missing_demo_cols].copy()

# 1) Drop rows/cols
rows_after_drop = len(missing_demo.dropna())
cols_after_drop = missing_demo.dropna(axis=1).shape[1]

# 2) Constant fill
const_fill = missing_demo.fillna("MISSING")

# 3) Median/mode imputation
num_cols_demo = [c for c in missing_demo.columns if to_numeric_loose(missing_demo[c]).notna().mean() >= 0.9]
cat_cols_demo = [c for c in missing_demo.columns if c not in num_cols_demo]

median_mode = missing_demo.copy()
for c in num_cols_demo:
    median_mode[c] = to_numeric_loose(median_mode[c])
num_imp = SimpleImputer(strategy="median")
cat_imp = SimpleImputer(strategy="most_frequent")

if num_cols_demo:
    median_mode[num_cols_demo] = num_imp.fit_transform(median_mode[num_cols_demo])
if cat_cols_demo:
    median_mode[cat_cols_demo] = cat_imp.fit_transform(median_mode[cat_cols_demo])

# 4) Missing indicator
indicator = MissingIndicator(features="all")
ind_train = indicator.fit_transform(missing_demo)
ind_cols = [f"is_missing_{c}" for c in missing_demo.columns]
ind_df = pd.DataFrame(ind_train, columns=ind_cols, index=missing_demo.index)

print("Rows after dropna:", rows_after_drop)
print("Columns after dropna(axis=1):", cols_after_drop)
show(ind_df, n=5)


In [ ]:
# 5) Group-wise imputation example: fill capital_gain by occupation median (train-fitted)
if {"capital_gain", "occupation"}.issubset(train_df.columns):
    cap_train = to_numeric_loose(train_df["capital_gain"])
    grp_map = (
        pd.DataFrame({"occupation": train_df["occupation"].astype(str), "capital_gain": cap_train})
        .groupby("occupation")["capital_gain"].median()
    )
    global_med = cap_train.median()

    def group_impute_cap(frame: pd.DataFrame) -> pd.Series:
        x = to_numeric_loose(frame["capital_gain"])
        grp = frame["occupation"].astype(str)
        return x.fillna(grp.map(grp_map)).fillna(global_med)

    cap_train_imp = group_impute_cap(train_df)
    cap_val_imp = group_impute_cap(val_df)
    cap_test_imp = group_impute_cap(test_df)

    print("Group-wise imputation done. Missing after fill (train/val/test):",
          int(cap_train_imp.isna().sum()), int(cap_val_imp.isna().sum()), int(cap_test_imp.isna().sum()))


In [ ]:
# 6) KNN and 7) Iterative imputation on numeric subset
impute_cols = [c for c in ["age", "hours_per_week", "capital_gain", "capital_loss", "education_num"] if c in train_df.columns]

train_num_imp = pd.DataFrame({c: to_numeric_loose(train_df[c]) for c in impute_cols}, index=train_df.index)
val_num_imp = pd.DataFrame({c: to_numeric_loose(val_df[c]) for c in impute_cols}, index=val_df.index)
test_num_imp = pd.DataFrame({c: to_numeric_loose(test_df[c]) for c in impute_cols}, index=test_df.index)

knn_imp = KNNImputer(n_neighbors=5)
train_knn = pd.DataFrame(knn_imp.fit_transform(train_num_imp), columns=impute_cols, index=train_num_imp.index)
val_knn = pd.DataFrame(knn_imp.transform(val_num_imp), columns=impute_cols, index=val_num_imp.index)
test_knn = pd.DataFrame(knn_imp.transform(test_num_imp), columns=impute_cols, index=test_num_imp.index)

iter_imp = IterativeImputer(random_state=SEED, max_iter=10)
train_iter = pd.DataFrame(iter_imp.fit_transform(train_num_imp), columns=impute_cols, index=train_num_imp.index)
val_iter = pd.DataFrame(iter_imp.transform(val_num_imp), columns=impute_cols, index=val_num_imp.index)
test_iter = pd.DataFrame(iter_imp.transform(test_num_imp), columns=impute_cols, index=test_num_imp.index)

print("KNN missing left in train:", int(train_knn.isna().sum().sum()))
print("Iterative missing left in train:", int(train_iter.isna().sum().sum()))


> **Do It Yourself**
>
> Define an imputation policy for at least 5 columns:
> - method
> - rationale
> - risk/tradeoff
>
> Hint: include at least one column where you would add a missingness indicator.

In [ ]:
imputation_policy = pd.DataFrame([
    {"column": "age", "method": "median", "why": "Numeric, robust baseline", "tradeoff": "Can reduce variance"},
    {"column": "hours_per_week", "method": "group-wise by occupation", "why": "Behavior depends on job type", "tradeoff": "Relies on category quality"},
    {"column": "capital_gain", "method": "iterative", "why": "Correlated with other numeric fields", "tradeoff": "Harder to explain"},
    {"column": "occupation", "method": "most_frequent + missing_indicator", "why": "Categorical + missing may be informative", "tradeoff": "Mode can hide minority patterns"},
    {"column": "native_country", "method": "constant=MISSING", "why": "Protects unseen/missing path", "tradeoff": "Creates extra category"},
])
show(imputation_policy, n=10)

# Build a practical imputed matrix for downstream sections.
train_work = mixed_train.copy()
val_work = mixed_val.copy()
test_work = mixed_test.copy()

# Add a few missing indicators from raw columns.
for col in ["age", "hours_per_week", "capital_gain", "occupation"]:
    if col in train_df.columns:
        train_work[f"{col}_is_missing"] = train_df[col].isna().astype(int)
        val_work[f"{col}_is_missing"] = val_df[col].isna().astype(int)
        test_work[f"{col}_is_missing"] = test_df[col].isna().astype(int)

# Fill numeric missing with train medians.
train_medians = train_work.median(numeric_only=True)
for col in train_work.columns:
    med = train_medians.get(col, 0.0)
    train_work[col] = train_work[col].fillna(med)
    val_work[col] = val_work[col].fillna(med)
    test_work[col] = test_work[col].fillna(med)

assert train_work.isna().sum().sum() == 0
assert val_work.isna().sum().sum() == 0
assert test_work.isna().sum().sum() == 0
print("Missingness handling check passed (no unintended NaN in working matrices).")


<a id="outliers"></a>
## Section 5 - Invalid Values and Outliers

We combine:
- rule-based cleaning,
- statistical clipping,
- model-based outlier detection (IsolationForest).

In [ ]:
def apply_rule_cleaning(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    if "age" in out.columns:
        age = to_numeric_loose(out["age"])
        bad_age = ((age < 0) | (age > 100)).fillna(False)
        out["age_invalid_flag"] = bad_age.astype(int)
        out["age"] = age.mask(bad_age)

    if "hours_per_week" in out.columns:
        hpw = to_numeric_loose(out["hours_per_week"])
        bad_hpw = ((hpw < 1) | (hpw > 120)).fillna(False)
        out["hours_invalid_flag"] = bad_hpw.astype(int)
        out["hours_per_week"] = hpw.mask(bad_hpw)

    if "capital_gain" in out.columns:
        cg = to_numeric_loose(out["capital_gain"])
        bad_cg = (cg < 0).fillna(False)
        out["capital_gain_invalid_flag"] = bad_cg.astype(int)
        out["capital_gain"] = cg.mask(bad_cg)

    return out

train_rules = apply_rule_cleaning(train_df)
val_rules = apply_rule_cleaning(val_df)
test_rules = apply_rule_cleaning(test_df)

rule_counts = {
    "train_bad_age": int(train_rules.get("age_invalid_flag", pd.Series(dtype=int)).sum()),
    "train_bad_hours": int(train_rules.get("hours_invalid_flag", pd.Series(dtype=int)).sum()),
    "train_bad_capital_gain": int(train_rules.get("capital_gain_invalid_flag", pd.Series(dtype=int)).sum()),
}
print(rule_counts)


In [ ]:
# Statistical clipping with train-fitted IQR bounds
train_after_outliers = train_work.copy()
val_after_outliers = val_work.copy()
test_after_outliers = test_work.copy()

clip_log = []
clip_cols = [c for c in train_after_outliers.columns if train_after_outliers[c].dtype.kind in "biufc"]

for col in clip_cols:
    q1 = train_after_outliers[col].quantile(0.25)
    q3 = train_after_outliers[col].quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr):
        continue
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    before_train = int(((train_after_outliers[col] < low) | (train_after_outliers[col] > high)).sum())

    train_after_outliers[col] = train_after_outliers[col].clip(low, high)
    val_after_outliers[col] = val_after_outliers[col].clip(low, high)
    test_after_outliers[col] = test_after_outliers[col].clip(low, high)

    clip_log.append({"column": col, "low": low, "high": high, "n_train_clipped": before_train})

clip_log = pd.DataFrame(clip_log).sort_values("n_train_clipped", ascending=False)
show(clip_log, n=10)

# Robust z-score counts (MAD-based)
robust_counts = []
for col in clip_cols[:20]:
    med = train_after_outliers[col].median()
    mad = (train_after_outliers[col] - med).abs().median()
    if mad == 0 or pd.isna(mad):
        continue
    rz = 0.6745 * (train_after_outliers[col] - med) / mad
    robust_counts.append({"column": col, "n_abs_rz_gt_3.5": int((rz.abs() > 3.5).sum())})

robust_counts = pd.DataFrame(robust_counts).sort_values("n_abs_rz_gt_3.5", ascending=False)
show(robust_counts, n=10)


In [ ]:
# Model-based detector example: IsolationForest
iso_cols = [c for c in train_after_outliers.columns if train_after_outliers[c].dtype.kind in "biufc"][:30]

iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=SEED,
)
iso.fit(train_after_outliers[iso_cols])

train_iso_flag = (iso.predict(train_after_outliers[iso_cols]) == -1).astype(int)
val_iso_flag = (iso.predict(val_after_outliers[iso_cols]) == -1).astype(int)
test_iso_flag = (iso.predict(test_after_outliers[iso_cols]) == -1).astype(int)

train_after_outliers["iso_outlier_flag"] = train_iso_flag
val_after_outliers["iso_outlier_flag"] = val_iso_flag
test_after_outliers["iso_outlier_flag"] = test_iso_flag

print("IsolationForest outlier share (train/val/test):",
      round(train_iso_flag.mean(), 4),
      round(val_iso_flag.mean(), 4),
      round(test_iso_flag.mean(), 4))


> **Do It Yourself**
>
> Create a per-feature policy table with one of:
> - drop row
> - clip/cap
> - set to missing + impute
> - keep + add flag
>
> Hint: prioritize rules for impossible values before statistical outlier methods.

In [ ]:
outlier_policy = pd.DataFrame([
    {"column": "age", "policy": "set impossible to NaN -> median impute", "reason": "Physical domain limits"},
    {"column": "hours_per_week", "policy": "clip + keep invalid flag", "reason": "Potential data-entry errors"},
    {"column": "capital_gain", "policy": "right-tail clip at IQR upper", "reason": "Extreme skew"},
    {"column": "occupation_te_safe", "policy": "keep as is", "reason": "Already smoothed encoded value"},
])
show(outlier_policy, n=10)

if "age" in train_rules.columns:
    age_clean = train_rules["age"]
    assert ((age_clean.dropna() >= 0) & (age_clean.dropna() <= 100)).all()

print("Outlier/invalid value checks passed.")


<a id="scaling"></a>
## Section 6 - Scaling and Numeric Transformations

We compare Standard, MinMax, and Robust scaling, then add log/power transforms for skewed features.

In [ ]:
scale_cols = [c for c in train_after_outliers.columns if train_after_outliers[c].dtype.kind in "biufc"]
scale_cols = [c for c in scale_cols if c not in ["iso_outlier_flag"]][:40]

std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
rb_scaler = RobustScaler()

train_std = pd.DataFrame(std_scaler.fit_transform(train_after_outliers[scale_cols]), columns=scale_cols, index=train_after_outliers.index)
val_std = pd.DataFrame(std_scaler.transform(val_after_outliers[scale_cols]), columns=scale_cols, index=val_after_outliers.index)

train_mm = pd.DataFrame(mm_scaler.fit_transform(train_after_outliers[scale_cols]), columns=scale_cols, index=train_after_outliers.index)
train_rb = pd.DataFrame(rb_scaler.fit_transform(train_after_outliers[scale_cols]), columns=scale_cols, index=train_after_outliers.index)

compare_col = scale_cols[0]
scaling_compare = pd.DataFrame({
    "original": train_after_outliers[compare_col].describe(),
    "standard": train_std[compare_col].describe(),
    "minmax": train_mm[compare_col].describe(),
    "robust": train_rb[compare_col].describe(),
})
show(scaling_compare, n=10)

# Choose robust as default working scaler for this notebook.
train_scaled_df = train_after_outliers.copy()
val_scaled_df = val_after_outliers.copy()
test_scaled_df = test_after_outliers.copy()

train_scaled_df[scale_cols] = rb_scaler.fit_transform(train_after_outliers[scale_cols])
val_scaled_df[scale_cols] = rb_scaler.transform(val_after_outliers[scale_cols])
test_scaled_df[scale_cols] = rb_scaler.transform(test_after_outliers[scale_cols])

assert np.isfinite(train_scaled_df[scale_cols].to_numpy()).all()
print("Scaling check passed.")


In [ ]:
# Log and power transforms on skewed features
skew_candidates = [c for c in ["capital_gain", "capital_loss", "hours_per_week"] if c in train_scaled_df.columns]

for c in skew_candidates:
    raw_train = train_after_outliers[c].clip(lower=0)
    train_scaled_df[f"{c}_log1p"] = np.log1p(raw_train)
    val_scaled_df[f"{c}_log1p"] = np.log1p(val_after_outliers[c].clip(lower=0))
    test_scaled_df[f"{c}_log1p"] = np.log1p(test_after_outliers[c].clip(lower=0))

if "capital_gain" in train_after_outliers.columns:
    pt = PowerTransformer(method="yeo-johnson")
    train_scaled_df["capital_gain_power"] = pt.fit_transform(train_after_outliers[["capital_gain"]]).ravel()
    val_scaled_df["capital_gain_power"] = pt.transform(val_after_outliers[["capital_gain"]]).ravel()
    test_scaled_df["capital_gain_power"] = pt.transform(test_after_outliers[["capital_gain"]]).ravel()

print("Added transformed columns:", [c for c in train_scaled_df.columns if c.endswith("_log1p") or c.endswith("_power")])


> **Do It Yourself**
>
> Pick one scaler for each numeric feature group and justify with distribution shape.
>
> Hint: use robust scaling when outliers are frequent.

In [ ]:
scale_recommendations = []
for c in scale_cols[:20]:
    s = train_after_outliers[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    outlier_rate = (((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()) if iqr > 0 else 0
    skew = s.skew() if s.nunique() > 2 else 0

    if outlier_rate > 0.02:
        rec = "RobustScaler"
    elif abs(skew) > 1.0:
        rec = "log/Power + StandardScaler"
    else:
        rec = "StandardScaler"

    scale_recommendations.append({"column": c, "outlier_rate": round(float(outlier_rate), 4), "skew": round(float(skew), 3), "recommended": rec})

show(pd.DataFrame(scale_recommendations), n=20)


<a id="engineering"></a>
## Section 7 - Feature Engineering (Advanced)

Include interactions, ratios, group aggregates, discretization, log transforms,
datetime features, polynomial expansion, and hashing trick.

In [ ]:
train_fe_df = train_scaled_df.copy()
val_fe_df = val_scaled_df.copy()
test_fe_df = test_scaled_df.copy()

# 1) Interaction + ratio features
if {"age", "hours_per_week"}.issubset(train_fe_df.columns):
    train_fe_df["age_x_hours"] = train_fe_df["age"] * train_fe_df["hours_per_week"]
    val_fe_df["age_x_hours"] = val_fe_df["age"] * val_fe_df["hours_per_week"]
    test_fe_df["age_x_hours"] = test_fe_df["age"] * test_fe_df["hours_per_week"]

if {"capital_gain", "hours_per_week"}.issubset(train_fe_df.columns):
    train_fe_df["capital_gain_per_hour"] = train_after_outliers["capital_gain"] / (train_after_outliers["hours_per_week"].abs() + 1)
    val_fe_df["capital_gain_per_hour"] = val_after_outliers["capital_gain"] / (val_after_outliers["hours_per_week"].abs() + 1)
    test_fe_df["capital_gain_per_hour"] = test_after_outliers["capital_gain"] / (test_after_outliers["hours_per_week"].abs() + 1)

# 2) Group aggregates learned on train
if {"occupation", "hours_per_week"}.issubset(train_df.columns):
    grp_hours = train_df.assign(hours_num=to_numeric_loose(train_df["hours_per_week"]))        .groupby(train_df["occupation"].astype(str))["hours_num"].mean()
    grp_target = train_df.groupby(train_df["occupation"].astype(str))[TARGET_BIN_COL].mean()

    for raw, out in [(train_df, train_fe_df), (val_df, val_fe_df), (test_df, test_fe_df)]:
        key = raw["occupation"].astype(str)
        out["occupation_mean_hours"] = key.map(grp_hours).fillna(grp_hours.mean())
        out["occupation_target_rate_train"] = key.map(grp_target).fillna(grp_target.mean())

# 3) Binning/discretization
if "age" in train_after_outliers.columns:
    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
    age_train_arr = train_after_outliers[["age"]].to_numpy(copy=True)
    age_val_arr = val_after_outliers[["age"]].to_numpy(copy=True)
    age_test_arr = test_after_outliers[["age"]].to_numpy(copy=True)
    train_fe_df["age_bin"] = kb.fit_transform(age_train_arr).ravel()
    val_fe_df["age_bin"] = kb.transform(age_val_arr).ravel()
    test_fe_df["age_bin"] = kb.transform(age_test_arr).ravel()

# 4) Datetime-derived features
if "record_written_at" in train_df.columns:
    for raw, out in [(train_df, train_fe_df), (val_df, val_fe_df), (test_df, test_fe_df)]:
        dt = pd.to_datetime(raw["record_written_at"], errors="coerce")
        out["record_month"] = dt.dt.month.fillna(0).astype(int)
        out["record_dayofweek"] = dt.dt.dayofweek.fillna(0).astype(int)
        out["record_hour"] = dt.dt.hour.fillna(0).astype(int)

# 5) Polynomial expansion (compact subset)
poly_base = [c for c in ["age", "hours_per_week", "education_num"] if c in train_after_outliers.columns]
if len(poly_base) >= 2:
    poly = PolynomialFeatures(degree=2, include_bias=False)
    tr_poly = poly.fit_transform(train_after_outliers[poly_base])
    va_poly = poly.transform(val_after_outliers[poly_base])
    te_poly = poly.transform(test_after_outliers[poly_base])

    poly_cols = [f"poly_{c}" for c in poly.get_feature_names_out(poly_base)]
    tr_poly_df = pd.DataFrame(tr_poly, columns=poly_cols, index=train_fe_df.index)
    va_poly_df = pd.DataFrame(va_poly, columns=poly_cols, index=val_fe_df.index)
    te_poly_df = pd.DataFrame(te_poly, columns=poly_cols, index=test_fe_df.index)

    # Keep only interaction and squared terms (skip raw duplicates).
    keep_poly = [c for c in poly_cols if " " in c.replace("poly_", "") or "^2" in c]
    train_fe_df = pd.concat([train_fe_df, tr_poly_df[keep_poly]], axis=1)
    val_fe_df = pd.concat([val_fe_df, va_poly_df[keep_poly]], axis=1)
    test_fe_df = pd.concat([test_fe_df, te_poly_df[keep_poly]], axis=1)

# 6) Hashing trick for high-cardinality / noisy categorical text-like values
if "native_country" in train_df.columns:
    hasher = FeatureHasher(n_features=8, input_type="dict")

    def hash_series(series: pd.Series, prefix: str) -> pd.DataFrame:
        tokens = series.fillna("MISSING").astype(str).tolist()
        records = [{tok: 1.0} for tok in tokens]
        mat = hasher.transform(records)
        cols = [f"{prefix}_{i}" for i in range(mat.shape[1])]
        return pd.DataFrame(mat.toarray(), columns=cols, index=series.index)

    train_hash = hash_series(train_df["native_country"], "hash_native_country")
    val_hash = hash_series(val_df["native_country"], "hash_native_country")
    test_hash = hash_series(test_df["native_country"], "hash_native_country")

    train_fe_df = pd.concat([train_fe_df, train_hash], axis=1)
    val_fe_df = pd.concat([val_fe_df, val_hash], axis=1)
    test_fe_df = pd.concat([test_fe_df, test_hash], axis=1)

# Final cleanup
for df in [train_fe_df, val_fe_df, test_fe_df]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

train_fe_medians = train_fe_df.median(numeric_only=True)
for col in train_fe_df.columns:
    med = train_fe_medians.get(col, 0.0)
    train_fe_df[col] = train_fe_df[col].fillna(med)
    val_fe_df[col] = val_fe_df[col].fillna(med)
    test_fe_df[col] = test_fe_df[col].fillna(med)

print("Feature-engineered shapes:", train_fe_df.shape, val_fe_df.shape, test_fe_df.shape)


> **Do It Yourself**
>
> Propose 3 engineered features and write one hypothesis per feature.
>
> Hint: each feature should map to a behavioral or business explanation.

In [ ]:
feature_hypotheses = pd.DataFrame([
    {"feature": "capital_gain_per_hour", "hypothesis": "High gain intensity per work-hour signals higher income class", "leakage_risk": "low"},
    {"feature": "occupation_mean_hours", "hypothesis": "Occupation-level workload baseline adds contextual prior", "leakage_risk": "low (train-only aggregate)"},
    {"feature": "record_dayofweek", "hypothesis": "Application workflow timing may correlate with process outcomes", "leakage_risk": "medium (process artifact)"},
])
show(feature_hypotheses, n=10)

num_arr = train_fe_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
assert np.isfinite(num_arr).all()
assert train_fe_df.isna().sum().sum() == 0
print("Feature engineering sanity checks passed.")


<a id="selection"></a>
## Section 8 - Feature Selection / Dimensionality Reduction (Qualitative)

No model-score benchmark here. We use qualitative selectors and interpretive outputs.

In [ ]:
# Use numeric matrix for selectors.
X_sel_train = train_fe_df.select_dtypes(include=[np.number]).copy()
X_sel_val = val_fe_df[X_sel_train.columns].copy()
X_sel_test = test_fe_df[X_sel_train.columns].copy()

# 1) Variance threshold
vt = VarianceThreshold(threshold=0.0)
vt.fit(X_sel_train)
vt_cols = X_sel_train.columns[vt.get_support()].tolist()

# 2) Correlation pruning
corr = X_sel_train[vt_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
remove_corr = [col for col in upper.columns if (upper[col] > 0.95).any()]
corr_pruned_cols = [c for c in vt_cols if c not in remove_corr]

# 3) Mutual information ranking
mi = mutual_info_classif(X_sel_train[corr_pruned_cols], y_train, random_state=SEED)
mi_rank = pd.DataFrame({"feature": corr_pruned_cols, "mi": mi}).sort_values("mi", ascending=False)

# 4) Model-based importance (illustrative only)
rf = RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1)
rf.fit(X_sel_train[corr_pruned_cols], y_train)
imp_rank = pd.DataFrame({
    "feature": corr_pruned_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

# 5) PCA projection
pca_input_cols = corr_pruned_cols[: min(80, len(corr_pruned_cols))]
std_for_pca = StandardScaler()
X_pca_train = std_for_pca.fit_transform(X_sel_train[pca_input_cols])

pca = PCA(n_components=min(10, X_pca_train.shape[1]), random_state=SEED)
pcs = pca.fit_transform(X_pca_train)

pca_var = pd.DataFrame({
    "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
})

show(mi_rank, n=15)
show(imp_rank, n=15)
show(pca_var, n=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pca_var["component"], pca_var["cumulative"], marker="o")
axes[0].set_title("PCA cumulative explained variance")
axes[0].set_xlabel("component")
axes[0].set_ylabel("cumulative variance")

scatter_idx = np.random.RandomState(SEED).choice(len(pcs), size=min(2000, len(pcs)), replace=False)
axes[1].scatter(pcs[scatter_idx, 0], pcs[scatter_idx, 1], c=y_train.iloc[scatter_idx], alpha=0.3, s=10)
axes[1].set_title("PCA projection (PC1 vs PC2)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
plt.tight_layout()
plt.show()

selected_features_v1 = corr_pruned_cols


> **Do It Yourself**
>
> Choose a candidate subset by combining at least two methods (e.g., MI + correlation pruning).
>
> Hint: keep interpretability in mind, not only ranking position.

In [ ]:
top_mi = mi_rank.head(20)["feature"].tolist()
top_imp = imp_rank.head(20)["feature"].tolist()

candidate_subset = sorted(set(top_mi).intersection(set(top_imp)))
if len(candidate_subset) < 10:
    candidate_subset = sorted(set(top_mi[:10] + top_imp[:10]))

print("Candidate subset size:", len(candidate_subset))
print("Candidate subset sample:", candidate_subset[:20])


<a id="ames-lab"></a>
## Section 9 - Mini-Lab A (Ames) - Open Ended

> **Do It Yourself**
>
> Explore and propose preprocessing actions for:
> - unit inconsistencies,
> - truncated/capped values,
> - implicit hierarchy extraction,
> - multiple rows per entity.
>
> Write your own checks first, then compare with the lightweight reference solution below.

In [ ]:
ames_lab = pd.read_csv("day1/generated/ames_housing_issues.csv")
print("Ames lab shape:", ames_lab.shape)
show(ames_lab, n=5)


In [ ]:
# Lightweight reference solution
lot_ratio = ames_lab["lot_area_reported"] / ames_lab["lot_area"]
unit_check = pd.DataFrame({
    "ratio": lot_ratio,
    "likely_sqm": lot_ratio.between(0.09, 0.095, inclusive="both"),
    "likely_sqft": lot_ratio.between(0.99, 1.01, inclusive="both"),
})

cap_value = ames_lab["reported_max_loan"].quantile(0.97)
capped_rows = ames_lab[ames_lab["reported_max_loan"] >= cap_value]

tax_split = ames_lab["property_taxonomy_flat"].astype(str).str.split(">", n=1, expand=True)
tax_split.columns = ["zone_level", "neighborhood_level"]

entity_dupes = ames_lab["property_id"].duplicated(keep=False).mean()

print("Likely sqm rows:", int(unit_check["likely_sqm"].sum()))
print("Likely sqft rows:", int(unit_check["likely_sqft"].sum()))
print("Cap threshold (97th pct):", round(float(cap_value), 2), "rows at/above:", len(capped_rows))
print("Share of duplicated property_id rows:", round(float(entity_dupes), 4))
show(pd.concat([ames_lab[["property_taxonomy_flat"]], tax_split], axis=1), n=8)


<a id="retail-lab"></a>
## Section 10 - Mini-Lab B (Retail) - Open Ended

> **Do It Yourself**
>
> Investigate:
> - data drift over time,
> - time misalignment / future leakage,
> - process artifact variables.
>
> Then compare with the lightweight reference solution.

In [ ]:
retail_lab = pd.read_csv("day1/generated/retail_panel_issues.csv")
retail_lab["date"] = pd.to_datetime(retail_lab["date"])
print("Retail lab shape:", retail_lab.shape)
show(retail_lab, n=5)


In [ ]:
# Lightweight reference solution
cutoff = pd.Timestamp("2023-10-01")
retail_lab["period"] = np.where(retail_lab["date"] < cutoff, "pre_policy", "post_policy")

drift_summary = retail_lab.groupby("period")["sales"].describe()

ret_sorted = retail_lab.sort_values(["store_id", "date"]).copy()
ret_sorted["inventory_t_plus_1"] = ret_sorted.groupby("store_id")["inventory_units"].shift(-1)
misalign_match = (
    ret_sorted["inventory_after_restock"].fillna(-9999)
    == ret_sorted["inventory_t_plus_1"].fillna(-9999)
).mean()

artifact_summary = retail_lab.groupby("workflow_route_code")["sales"].describe()

print("Drift summary (sales by period):")
display(drift_summary)
print("Time-misalignment match rate inventory_after_restock == next-day inventory_units:", round(float(misalign_match), 4))
print("Process artifact summary (sales by workflow_route_code):")
display(artifact_summary)


<a id="checklist"></a>
## Section 11 - End-of-Notebook Preprocessing Checklist

> **Do It Yourself**
>
> Fill this checklist for one dataset before moving to modeling:

> - [ ] Split strategy is deployment-consistent (time/group aware if needed).
> - [ ] Column typing is validated (numeric/categorical/date/text/process).
> - [ ] Leakage columns are excluded from features.
> - [ ] Categorical encoding choices documented by feature type.
> - [ ] Missingness policy documented (with indicators where useful).
> - [ ] Invalid/outlier policy documented (rule/statistical/model-based).
> - [ ] Scaling/transform choices documented by model sensitivity.
> - [ ] Engineered features have explicit business hypotheses.
> - [ ] Feature selection logic combines at least 2 methods.
> - [ ] Deployment assumptions are explicitly written.


In [ ]:
student_checklist_template = {
    "dataset": "adult_income_issues.csv",
    "split_strategy": "use provided split + train/val inside train",
    "typing_done": False,
    "leakage_checked": False,
    "encoding_policy_done": False,
    "missingness_policy_done": False,
    "outlier_policy_done": False,
    "scaling_policy_done": False,
    "feature_hypotheses_done": False,
    "selection_policy_done": False,
    "deployment_assumptions_written": False,
}

student_checklist_template


In [ ]:
reference_checklist = {
    "dataset": "adult_income_issues.csv",
    "split_strategy": "Use provided split to preserve unseen test categories; derive val from train only.",
    "typing_done": True,
    "leakage_checked": True,
    "encoding_policy_done": True,
    "missingness_policy_done": True,
    "outlier_policy_done": True,
    "scaling_policy_done": True,
    "feature_hypotheses_done": True,
    "selection_policy_done": True,
    "deployment_assumptions_written": True,
}

pd.DataFrame(reference_checklist, index=[0])


<a id="acceptance"></a>
## Acceptance Checks
These checks validate the Day 1 preprocessing/feature-engineering workflow end-to-end.

In [ ]:
# 1) Split integrity
assert len(set(train_df[ID_COLS[0]]) & set(val_df[ID_COLS[0]])) == 0
assert len(set(train_df[ID_COLS[0]]) & set(test_df[ID_COLS[0]])) == 0

# 2) Leakage controls
for leak_col in LEAKAGE_COLS:
    assert leak_col not in train_fe_df.columns, f"Leakage col found in features: {leak_col}"

# 3) Encoding correctness
encoded_cols = list(X_train_ohe_df.columns) + list(X_train_freq.columns)
if "occupation_te_safe" in mixed_train.columns:
    encoded_cols.append("occupation_te_safe")
encoded_num = mixed_train[encoded_cols].to_numpy(dtype=float)
assert np.isfinite(encoded_num).all()
assert mixed_train.shape[1] == mixed_val.shape[1] == mixed_test.shape[1]

# 4) Missingness handling
assert train_work.isna().sum().sum() == 0
assert val_work.isna().sum().sum() == 0
assert test_work.isna().sum().sum() == 0

# 5) Outlier/invalid handling
if "age" in train_rules.columns:
    assert ((train_rules["age"].dropna() >= 0) & (train_rules["age"].dropna() <= 100)).all()
assert "iso_outlier_flag" in train_after_outliers.columns

# 6) Feature engineering sanity
num_arr = train_fe_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
assert np.isfinite(num_arr).all()

# 7) Feature selection outputs
assert len(candidate_subset) > 0
assert "explained_variance_ratio" in pca_var.columns

# 8) Reproducibility hooks
assert SEED == 42

print("All acceptance checks passed.")
print("Notebook is ready for Day 1 delivery.")
